# ODIL — Discrete-Loss Solver (Karnakov et al. 2024)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cmhobbs96/pift-od-il-inverse-problems/blob/main/examples/03_odil_solver.ipynb)

**Optimizing a Discrete Loss (ODIL)** ([Karnakov et al. 2024](https://doi.org/10.1093/pnasnexus/pgae005))
solves PDEs by minimizing a sum of squared finite-difference (FD) residuals at grid nodes.
For the 1D Poisson equation $-u'' = f$ on a uniform grid with $N$ interior nodes and spacing
$h = 1/(N+1)$, the discrete residual at node $i$ is
$$r_i = \frac{-u_{i-1} + 2u_i - u_{i+1}}{h^2} - f_i.$$
Minimizing $\|r\|^2$ subject to boundary constraints gives the MAP estimate.

We compare two optimizers:
- **Gauss-Newton (GN):** exploits the linear structure of the residual; converges in $\approx 2$
  iterations for this linear problem.
- **L-BFGS:** quasi-Newton method; converges more slowly but requires no Jacobian.

ODIL solutions will be used as warm-starts for PIFT in notebook 04.

In [ ]:
%pip install -q git+https://github.com/cmhobbs96/pift-od-il-inverse-problems.git

import time
import jax
jax.config.update('jax_enable_x64', True)
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

print('JAX backend:', jax.default_backend(), '| devices:', jax.devices())

from core.odil import odil_solve_poisson_1d
from pipelines.common import forcing, phi_true
from pipelines.phase_a import run_phase_a_odil

def show_fig(fig, dpi=120):
    import tempfile
    from IPython.display import Image, display
    with tempfile.NamedTemporaryFile(suffix='.png', delete=False) as f:
        fig.savefig(f.name, dpi=dpi, bbox_inches='tight')
        plt.close(fig)
        display(Image(f.name))

## Configuration

In [ ]:
CONFIG = {
    'seed':      7,        # reproducibility                                    [0, 2**31-1]
    'n_grid':    257,      # interior grid nodes (power-of-2 + 1 recommended)   [32, 4096]
    'max_iter':  50,       # maximum GN / L-BFGS iterations                     [5, 500]
    'bc_weight': 1e6,      # penalty weight for Dirichlet BCs                   [1e2, 1e9]
    'n_obs':     28,       # noisy observations for run_phase_a_odil            [4, 200]
    'noise_std': 0.08,     # observation noise sigma                            [0.0, 0.5]
    'tol':       1e-8,     # convergence tolerance on grad norm                 [1e-12, 1e-4]
}

## Gauss-Newton & L-BFGS Solutions

Both solvers minimize the same objective. We generate noisy observations, run both methods,
time them, and report iterations, wall time, and L2 error against the known truth.

In [ ]:
rng = np.random.default_rng(CONFIG['seed'])
x_int = np.linspace(0, 1, CONFIG['n_grid'] + 2)[1:-1]   # interior nodes
x_full = np.linspace(0, 1, CONFIG['n_grid'] + 2)         # full grid incl. BCs

# Ground truth on interior nodes
u_true = phi_true(x_int)
f_vals = forcing(x_int)

# --- Gauss-Newton ---
t0 = time.perf_counter()
gn_result = odil_solve_poisson_1d(
    forcing_vals=f_vals,
    method='gauss_newton',
    max_iter=CONFIG['max_iter'],
    bc_weight=CONFIG['bc_weight'],
    tol=CONFIG['tol'],
)
t_gn = time.perf_counter() - t0

# --- L-BFGS ---
t0 = time.perf_counter()
lbfgs_result = odil_solve_poisson_1d(
    forcing_vals=f_vals,
    method='lbfgs',
    max_iter=CONFIG['max_iter'],
    bc_weight=CONFIG['bc_weight'],
    tol=CONFIG['tol'],
)
t_lbfgs = time.perf_counter() - t0

l2_gn    = float(np.sqrt(np.mean((gn_result['solution']    - u_true) ** 2)))
l2_lbfgs = float(np.sqrt(np.mean((lbfgs_result['solution'] - u_true) ** 2)))

print(f'Gauss-Newton:  iters={gn_result["n_iter"]:3d}  '
      f'converged={gn_result["converged"]}  '
      f'time={t_gn*1000:.1f} ms  L2={l2_gn:.2e}')
print(f'L-BFGS:        iters={lbfgs_result["n_iter"]:3d}  '
      f'converged={lbfgs_result["converged"]}  '
      f'time={t_lbfgs*1000:.1f} ms  L2={l2_lbfgs:.2e}')

# --- Two-panel figure ---
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Left: solution overlay
axes[0].plot(x_int, u_true,                  'k-',  lw=2,   label='Truth')
axes[0].plot(x_int, gn_result['solution'],   'b--', lw=1.5, label=f'GN (L2={l2_gn:.1e})')
axes[0].plot(x_int, lbfgs_result['solution'],'r:',  lw=1.5, label=f'L-BFGS (L2={l2_lbfgs:.1e})')
axes[0].set_xlabel('x')
axes[0].set_ylabel('u(x)')
axes[0].set_title('ODIL Solutions vs Truth')
axes[0].legend()

# Right: GN loss history
gn_loss = np.array(gn_result.get('loss_history', []))
if len(gn_loss) > 0:
    axes[1].semilogy(gn_loss, 'b-o', markersize=4, label='GN loss')
    axes[1].set_xlabel('Iteration')
    axes[1].set_ylabel('Loss')
    axes[1].set_title('GN Loss History')
    axes[1].legend()
else:
    lbfgs_loss = np.array(lbfgs_result.get('loss_history', []))
    if len(lbfgs_loss) > 0:
        axes[1].semilogy(lbfgs_loss, 'r-s', markersize=3, label='L-BFGS loss')
        axes[1].set_xlabel('Iteration')
        axes[1].set_ylabel('Loss')
        axes[1].set_title('L-BFGS Loss History')
        axes[1].legend()
    else:
        axes[1].text(0.5, 0.5, 'No loss history available',
                    ha='center', va='center', transform=axes[1].transAxes)

fig.tight_layout()
show_fig(fig)

## Runner Integration

The `run_phase_a_odil` pipeline wraps ODIL with observation data generation, run management,
and a standardized results contract. We verify it returns the expected keys and L2 metric.

In [ ]:
pipeline_cfg = {
    'seed':      CONFIG['seed'],
    'n_obs':     CONFIG['n_obs'],
    'noise_std': CONFIG['noise_std'],
    'n_grid':    CONFIG['n_grid'],
    'max_iter':  CONFIG['max_iter'],
    'bc_weight': CONFIG['bc_weight'],
    'method':    'gauss_newton',
}

res = run_phase_a_odil(config=pipeline_cfg, save_outputs=False)

print('Status:     ', res['status'])
print(f'Iterations: {res["n_iter"]}')
print(f'Converged:  {res["converged"]}')
print(f'Method:     {res["method"]}')
print(f'Runtime:    {res["runtime_s"]*1000:.1f} ms')
print(f'L2 error:   {res["l2_error"]:.4e}')

# Quick solution plot
x_plot = np.linspace(0, 1, CONFIG['n_grid'] + 2)[1:-1]
u_truth_plot = phi_true(x_plot)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(x_plot, u_truth_plot,      'k-',  lw=2,   label='Truth')
ax.plot(x_plot, res['solution'],   'b--', lw=1.8, label=f'ODIL GN (L2={res["l2_error"]:.1e})')
if 'x_obs' in res and 'y_obs' in res:
    ax.scatter(res['x_obs'], res['y_obs'], s=20, color='tomato', zorder=5,
               label=f'Obs (n={CONFIG["n_obs"]})')
ax.set_xlabel('x')
ax.set_ylabel('u(x)')
ax.set_title(f'ODIL GN via Runner  |  n_grid={CONFIG["n_grid"]}')
ax.legend()
show_fig(fig)

## Interpretation

**Expected results** (default CONFIG):

- **Gauss-Newton:** converges in $\approx 2$ iterations because the Poisson residual is
  linear in $u$, so the Gauss-Newton step is exact. Wall time is $< 1$ ms on CPU,
  $< 0.1$ ms on GPU.
- **L-BFGS:** requires $\approx 10$–30 iterations to reach comparable accuracy because it
  cannot exploit the exact Hessian structure. Achieves L2 $\approx 10^{-5}$ at convergence.
- **GN vs L-BFGS accuracy:** both should reach L2 well below $10^{-3}$ with `n_grid=257`;
  GN is faster and more accurate for this linear problem.
- The ODIL solution is a **point estimate** (MAP), carrying no uncertainty. Notebook 04
  uses it as a warm-start for PIFT to combine ODIL speed with PIFT uncertainty quantification.